# lightgbm — r2_full_causal
Independent validation-screen run; seed 42. Validation selects class weighting/calibration and threshold policy. Final 40% is prediction-only.

In [ ]:
import base64, gzip, json, sys
from pathlib import Path
bundle=json.loads(gzip.decompress(base64.b64decode("...")))
pkg=Path("/kaggle/working/fraudlab_pkg/fraudlab"); pkg.mkdir(parents=True,exist_ok=True)
for name,source in bundle.items(): (pkg/name).write_text(source,encoding="utf-8")
sys.path.insert(0,str(pkg.parent))
print("Project modules loaded",len(bundle))


In [ ]:
from pathlib import Path
import json, time
from fraudlab.pipeline import train_recipe_model_from_artifacts
from fraudlab.config import ExperimentConfig, BanditConfig
MODEL_NAME="lightgbm"
RECIPE="r2_full_causal"
SEED=42
root=Path("/kaggle/input")
files=list(root.rglob("ieee_cis_features.parquet"))
tx=[p for p in files if p.parent.name=="transaction"]
ident=[p for p in files if p.parent.name=="identity"]
if not tx: raise FileNotFoundError("Attach completed preprocessing notebook output with transaction artifact")
config=ExperimentConfig(random_seed=SEED,bandit=BanditConfig(seed=SEED))
out=Path("/kaggle/working/outputs/model_grid")/(MODEL_NAME+"__"+RECIPE+"__seed_"+str(SEED))
started=time.time()
result=train_recipe_model_from_artifacts(tx[0],ident[0] if ident else None,RECIPE,MODEL_NAME,out,config)
record={"model":MODEL_NAME,"recipe":RECIPE,"seed":SEED,"runtime_seconds":time.time()-started,"result":result}
(out/"run_record.json").write_text(json.dumps(record,indent=2,default=str),encoding="utf-8")
print(json.dumps({"model":MODEL_NAME,"recipe":RECIPE,"runtime_seconds":round(time.time()-started,1),"validation":result["validation_metrics"],"validation_cost":result["validation_total_cost"]},indent=2))
